In [ ]:
from data import fetch_prices, compute_asset_returns, compute_portfolio_returns
from features import build_feature_frame, chronological_split
from models import historical_baseline, evaluate, train_and_select
from risk_metrics import compute_all_metrics
import matplotlib.pyplot as plt

In [ ]:
TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "SPY"]
WEIGHTS = {"AAPL": 0.25, "MSFT": 0.25, "GOOGL": 0.2, "AMZN": 0.15, "SPY": 0.15}
prices = fetch_prices(TICKERS, "2015-01-01", "2025-01-01")
prices.tail()

In [ ]:
asset_returns = compute_asset_returns(prices)
portfolio_returns = compute_portfolio_returns(asset_returns, WEIGHTS)
portfolio_returns.plot(title="Portfolio Daily Returns")

In [ ]:
print(compute_all_metrics(portfolio_returns))

In [ ]:
df, feature_cols = build_feature_frame(portfolio_returns)
train, val, test = chronological_split(df)
len(train), len(val), len(test)

In [ ]:
best_model, best_name, val_results, scaler = train_and_select(train, train["target_vol"], val, val["target_vol"], feature_cols)
best_name, val_results

In [ ]:
X_test = scaler.transform(test[feature_cols])
ml_preds = best_model.predict(X_test)
baseline_preds = historical_baseline(test)
print("ML:", evaluate(test["target_vol"], ml_preds))
print("Baseline:", evaluate(test["target_vol"], baseline_preds))

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(test.index, test["target_vol"].values, label="Actual")
plt.plot(test.index, ml_preds, label="ML")
plt.plot(test.index, baseline_preds.values, label="Baseline")
plt.legend()
plt.show()